In [1]:
import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
from matplotlib import pylab
import os
import sys
anndata2ri.activate()
import yaml
%reload_ext rpy2.ipython

from scipy.sparse import csr_matrix, isspmatrix


/data/projects/spatialTX/.local/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/tmp/ipykernel_577905/1056915137.py:8: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


In [2]:
pylab.rcParams['figure.figsize'] = (9, 9)
homeDir = os.getenv("HOME")
sys.path.insert(1, homeDir+"/utils/")
from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *
from spatialUtils import *
from _DEAplots import *
from _Aggregation import *
from _plotting import *


nThreads = 10
import os

os.environ["OMP_NUM_THREADS"] = f"{nThreads}"
os.environ["OPENBLAS_NUM_THREADS"] = f"{nThreads}"
os.environ["MKL_NUM_THREADS"] = f"{nThreads}"
os.environ["BLIS_NUM_THREADS"] = f"{nThreads}"
os.environ["VECLIB_MAXIMUM_THREADS"] = f"{nThreads}"
os.environ["MKL_DYNAMIC"] = "FALSE"


/usr/lib/python3.10/importlib/__init__.py:126: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/local/lib/python3.10/dist-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [16]:
import ipynbname
import nbconvert.exporters
from nbconvert.preprocessors import TagRemovePreprocessor
import os


try:
    nb_name = ipynbname.name()
except:
    nb_name = "".join(os.path.basename(globals()['__vsc_ipynb_file__']))

print(nb_name)

5_GSEA_per_timepoint


In [3]:
with open(homeDir+"/utils/config.yaml", 'r') as f:
    analysis_params = yaml.safe_load(f)["analysisParams"]
print(analysis_params)

{'n_neighbors': 40, 'n_neighbors_spatial': 6, 'n_pcs': 20, 'leidenRes': 0.6, 'randomState': 0}


# Load data

In [4]:
adata = sc.read_h5ad("./MelanomaCoCulture_processed.h5ad")

In [5]:
adata.X.max()

6.673043

In [6]:
from transferUtils import *

export_anndata_minimal(
    adata=adata,
    out_dir=f"./SCgseaReady",
    base=f"SCgseaReady",
    layer="counts",                 # ignored for data, used only for shape checking
    obsm_key="X_pca",
    replace_counts_with_zeros=False, # <- zeros
)

Saved counts: SCgseaReady/SCgseaReady_counts.npz  (shape=(9824, 36601), nnz=30130078)
Saved coords: SCgseaReady/SCgseaReady_coords.tsv  (n_cols=50, preview=['X_pca_1', 'X_pca_2', 'X_pca_3', 'X_pca_4', 'X_pca_5']...)
Saved var:    SCgseaReady/SCgseaReady_var.tsv   (rows=36601, cols=10)
Saved obs:    SCgseaReady/SCgseaReady_obs.tsv   (rows=9824, cols=23)


{'counts': PosixPath('SCgseaReady/SCgseaReady_counts.npz'),
 'coords': PosixPath('SCgseaReady/SCgseaReady_coords.tsv'),
 'var': PosixPath('SCgseaReady/SCgseaReady_var.tsv'),
 'obs': PosixPath('SCgseaReady/SCgseaReady_obs.tsv')}

In [12]:
%%R  -i homeDir -o CtMarkers96HRS -o CtMarkers2Weeks -o CtMarkers1Month
library(Seurat)
library(dplyr)

source(paste0(homeDir, "/utils/transferUtils.R"))

# If needed, pin Python with SciPy before calling:
# reticulate::use_python("/usr/bin/python", required = TRUE)

sce <- load_export_as_sce(
  out_dir = "./SCgseaReady",
  base = "SCgseaReady",
  reduced_name = "X_pca",
  add_coords_to_reduced = TRUE,
  include_coords_in_coldata = TRUE,
  counts_transpose = TRUE,
  sep = "\t"
)

SeuratObject <- as.Seurat(sce, counts = "counts", data = "counts")

# 96h vs preexposure
Contrast96Hrs <- subset(
  SeuratObject,
  subset = condition %in% c("Melanoma", "Melanoma_96Hrs_CoColture")
)
Contrast96Hrs <- NormalizeData(Contrast96Hrs)
Idents(Contrast96Hrs) <- "condition"

CtMarkers96HRS <- FindMarkers(
  Contrast96Hrs,
  ident.1 = "Melanoma_96Hrs_CoColture",
  ident.2 = "Melanoma",
  only.pos = FALSE,
  min.pct = 0.01,
  logfc.threshold = 0,
  min.cells.feature = 0,
  min.cells.group = 0
)
CtMarkers96HRS$gene <- rownames(CtMarkers96HRS)
CtMarkers96HRS$comparison <- "Melanoma_96Hrs_CoColture_vs_Melanoma"

# 2 weeks vs preexposure
Contrast2Weeks <- subset(
  SeuratObject,
  subset = condition %in% c("Melanoma", "Melanoma_2Weeks_CoColture")
)
Contrast2Weeks <- NormalizeData(Contrast2Weeks)
Idents(Contrast2Weeks) <- "condition"

CtMarkers2Weeks <- FindMarkers(
  Contrast2Weeks,
  ident.1 = "Melanoma_2Weeks_CoColture",
  ident.2 = "Melanoma",
  only.pos = FALSE,
  min.pct = 0.01,
  logfc.threshold = 0,
  min.cells.feature = 0,
  min.cells.group = 0
)
CtMarkers2Weeks$gene <- rownames(CtMarkers2Weeks)
CtMarkers2Weeks$comparison <- "Melanoma_2Weeks_CoColture_vs_Melanoma"

# 1 month vs preexposure
Contrast1month <- subset(
  SeuratObject,
  subset = condition %in% c("Melanoma", "Melanoma_1Month_CoColture")
)
Contrast1month <- NormalizeData(Contrast1month)
Idents(Contrast1month) <- "condition"

CtMarkers1Month <- FindMarkers(
  Contrast1month,
  ident.1 = "Melanoma_1Month_CoColture",
  ident.2 = "Melanoma",
  only.pos = FALSE,
  min.pct = 0.01,
  logfc.threshold = 0,
  min.cells.feature = 0,
  min.cells.group = 0
)
CtMarkers1Month$gene <- rownames(CtMarkers1Month)
CtMarkers1Month$comparison <- "Melanoma_1Month_CoColture_vs_Melanoma"

Loaded SCE: 36601 features x 9824 obs | nnz=30130078 | colData=73 cols | rowData=10 cols | reducedDim='X_pca' (50 cols)
Performing log-normalization
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
Performing log-normalization
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
Performing log-normalization
0%   10   20   30   40   50   60   70   80   90   100%
[----|----|----|----|----|----|----|----|----|----|
**************************************************|
In addition: Warning message:
Keys should be one or more alphanumeric characters followed by an underscore, setting key from X_pca_ to Xpca_ 


In [13]:
pd.concat([CtMarkers96HRS, CtMarkers2Weeks, CtMarkers1Month]).reset_index()

,index,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,gene,comparison
0,NMT1,0.000000,2.208422,0.878,0.245,0.0,NMT1,Melanoma_96Hrs_CoColture_vs_Melanoma
1,C8orf33,0.000000,2.087107,0.820,0.232,0.0,C8orf33,Melanoma_96Hrs_CoColture_vs_Melanoma
2,NDUFA3,0.000000,1.774954,0.894,0.313,0.0,NDUFA3,Melanoma_96Hrs_CoColture_vs_Melanoma
3,EIF3J,0.000000,1.777096,0.868,0.309,0.0,EIF3J,Melanoma_96Hrs_CoColture_vs_Melanoma
4,RMRP,0.000000,4.144745,0.625,0.069,0.0,RMRP,Melanoma_96Hrs_CoColture_vs_Melanoma
...,...,...,...,...,...,...,...,...
47363,WDR18,0.997332,0.356268,0.129,0.136,1.0,WDR18,Melanoma_1Month_CoColture_vs_Melanoma
47364,CCNQ,0.997350,0.281671,0.144,0.153,1.0,CCNQ,Melanoma_1Month_CoColture_vs_Melanoma
47365,C16orf72,0.997723,0.314595,0.072,0.074,1.0,C16orf72,Melanoma_1Month_CoColture_vs_Melanoma
47366,WRAP53,0.997937,0.492630,0.058,0.060,1.0,WRAP53,Melanoma_1Month_CoColture_vs_Melanoma


In [19]:
combined = pd.concat([CtMarkers96HRS, CtMarkers2Weeks, CtMarkers1Month]).reset_index()


saveDir = f"DEGS_{nb_name}_SeuratMethod_GSEAready"
os.makedirs(saveDir, exist_ok=True)

for comparison in combined["comparison"].unique():
    localDegs = combined[combined["comparison"] == comparison].copy()  
    localDegs.to_excel(os.path.join(saveDir, f"DEGS_{comparison}_.xlsx"), index=False)